### Story - 2 : Can the FED Control Inflation and Maintain Full Employment

In [ ]:
import requests
import json
import prettytable
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import datetime
from functools import reduce
pio.renderers.default = "notebook"

### API requests to retrieve data from BLS
System only allows API retrieval up to 10 years of data.

#### To obtain the **CPI** and the **unemployment rate** from 1999-2024
#### We can split the API requests into 3 requests through a loop:

In [2]:
headers = {'Content-type': 'application/json'}
year_ranges = [(1999, 2008), (2009, 2018), (2019, 2024)]
series_ids = ['CUUR0000SA0L1E', 'LNS14000000']

all_data = []

for i, (start, end) in enumerate(year_ranges, start=1):
    data = json.dumps({
        'seriesid': series_ids,
        'startyear': str(start),
        'endyear': str(end)
    })

    #requests:
    p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
    json_data = p.json()
    
    for series in json_data['Results']['series']:
        seriesId = series['seriesID']
        for item in series['data']:
            year = item['year']
            period = item['period']
            value = item['value']
            footnotes= ",".join(
                [f['text'] for f in item['footnotes'] if f]
            )
            if 'M01' <= period <= 'M12':
                all_data.append({
                    'series_id': seriesId,
                    'year': year,
                    'period': period,
                    'value': value,
                    'footnotes': footnotes
                })

#Convert to pandas:
df = pd.DataFrame(all_data)

df.to_csv('bls.csv', index = False)
   

### To obtain the FRED rate through the API:

In [6]:
import requests
import pandas as pd
key = 'ffe6dec1e433657d45d1cdd6362aa518'

url = 'https://api.stlouisfed.org/fred/series/observations?'

params = {
    'series_id': 'FEDFUNDS',
    'file_type': 'json',
    'api_key': key,
    'observation_start': '1999-01-01',
    'observation_end': '2024-12-01',
    'frequency': 'm'
}

response = requests.get(url, params=params)
data = response.json()

#Import to pandas dataframe:
FED_df = pd.DataFrame(data['observations'])
FED_df = FED_df[['date', 'value']]
FED_df.rename(columns={'value': 'rate'}, inplace=True)
FED_df.head()

,date,rate
0,1999-01-01,4.63
1,1999-02-01,4.76
2,1999-03-01,4.81
3,1999-04-01,4.74
4,1999-05-01,4.74


### Data cleanings:

In [7]:
#To separate the CPI data from the unemployment data:
df_cpi = df[df['series_id'].str.startswith('CUU')]

df_unemply = df[df['series_id'].str.startswith('LNS')]

#Convert the months into months only:
df_cpi = df_cpi[['year', 'period', 'value']]
df_unemply = df_unemply[['year', 'period', 'value']]

#Rename columns:
df_cpi.rename(columns={'value':'cpi', 'period':'month'}, inplace = True)
df_unemply.rename(columns={'value':'unemply_rate', 'period':'month'}, inplace = True)

#Remove the 'M' in the month column:
df_cpi['month'] = df_cpi['month'].str.replace('M', '')
df_unemply['month'] = df_unemply['month'].str.replace('M', '')

#Add a day column then combine and convert the date column:
df_cpi['day'] = 1
df_unemply['day'] = 1

df_cpi['date'] = pd.to_datetime(df_cpi[['year', 'month', 'day']])
df_unemply['date'] =  pd.to_datetime(df_unemply[['year', 'month', 'day']])

df_cpi = df_cpi[['date', 'cpi']]
df_unemply = df_unemply[['date', 'unemply_rate']]

#use only the date and value columns and format the date column into date format:
FED_df['date'] = pd.to_datetime(FED_df['date'])
FED_df.set_index('date', inplace=True)

#Covert rate column to float:
FED_df['rate'] = pd.to_numeric(FED_df['rate'], errors = 'coerce')

#Merge the three files into a tidy format:
dfs = [df_cpi, df_unemply, FED_df]
df_final = reduce(lambda left, right: pd.merge(left, right, on='date'), dfs)

df_final.to_csv('story2_final.csv')
df_final.head()


,date,cpi,unemply_rate,rate
0,2008-12-01,216.100,7.3,0.16
1,2008-11-01,216.690,6.8,0.39
2,2008-10-01,217.023,6.5,0.97
3,2008-09-01,216.862,6.1,1.81
4,2008-08-01,216.476,6.1,2.00


In [ ]:
df_final_long = pd.melt(df_final, id_vars ='date', var_name = 'data_type', value_name = 'value')
df_final_long = df_final_long.sort_values('date')
df_final_long.head()

,date,data_type,value
0,2008-12-01,cpi,216.100
1,2008-11-01,cpi,216.690
2,2008-10-01,cpi,217.023
3,2008-09-01,cpi,216.862
4,2008-08-01,cpi,216.476


In [33]:
df_final_long.describe

<bound method NDFrame.describe of           date     data_type    value
119 1999-01-01           cpi    175.3
431 1999-01-01  unemply_rate      4.3
743 1999-01-01          rate     4.63
118 1999-02-01           cpi    175.7
742 1999-02-01          rate     4.76
..         ...           ...      ...
241 2024-11-01           cpi  321.947
553 2024-11-01  unemply_rate      4.2
552 2024-12-01  unemply_rate      4.1
864 2024-12-01          rate     4.48
240 2024-12-01           cpi  322.007

[936 rows x 3 columns]>

In [36]:

fig = px.line(df_final_long,
        x = 'date',
        y = 'value',
        facet_row = 'data_type',        
        title = 'Overall economy dynamics from 1999 - 2024')
fig.update_yaxes(matches=None)
for i, d_type in enumerate(df_final_long['data_type'].unique()):
    fig.layout.annotations[i]['text'] = d_type

fig.show()


In [32]:
px.line(df_final_long[df_final_long['data_type']=='cpi'],
        x = 'date',
        y = 'value')